# 🌐 Notebook 3: Phi Accrual in a Real Cluster

In notebooks 1 and 2 we monitored a single node. Real systems monitor **many** nodes at once, and the answer to *"is this node up?"* feeds into **decisions**:

- Routing: *stop sending reads to a node we suspect.*
- Membership: *evict a node from the cluster after high confidence.*
- Leader fencing: *wait for very high confidence before electing a new leader.*

In this notebook we simulate a tiny 5-node cluster, inject nasty real-world events (GC pauses, network partition, a real crash), and show how a single phi detector per node gives you **tunable, per-purpose decisions** from the same underlying signal.

This is how **Cassandra**, **Akka Cluster**, and **ScyllaDB** actually use phi in production.

## 1. Re-use the detector from notebook 2


In [ ]:
import math, random
from collections import deque

class PhiAccrual:
    def __init__(self, window=200, min_std=0.1, warmup=10):
        self.intervals = deque(maxlen=window)
        self.last = None
        self.min_std = min_std
        self.warmup = warmup  # wait for real statistics before reporting suspicion

    def heartbeat(self, t):
        if self.last is not None:
            self.intervals.append(t - self.last)
        self.last = t

    def phi(self, now):
        if self.last is None or len(self.intervals) < self.warmup:
            return 0.0
        mean = sum(self.intervals) / len(self.intervals)
        var  = sum((x - mean) ** 2 for x in self.intervals) / len(self.intervals)
        std  = max(math.sqrt(var), self.min_std)
        delta = now - self.last
        z = (delta - mean) / math.sqrt(2) / std
        p = max(0.5 * math.erfc(z), 1e-20)
        return -math.log10(p)

## 2. Simulate a 5-node cluster with real-world nastiness

- **node-A** – healthy the whole time.
- **node-B** – suffers a **1-second stop-the-world GC pause** at t=15s (alive, but silent).
- **node-C** – a **transient network partition** from t=20s lasting 1.8s.
- **node-D** – **really crashes** at t=25s and never comes back.
- **node-E** – chronically **slow** — mean gap 1.5s, higher jitter — but alive.

Each node heartbeats every ~1s (except node-E). Our monitor runs one `PhiAccrual` per node.

In [ ]:
random.seed(42)
TOTAL = 40.0

def stream(interval=1.0, jitter=0.3, pauses=(), dead_at=None, total=TOTAL):
    """Generate heartbeat timestamps. `pauses` is a list of (start, duration) windows."""
    beats, t = [], 0.0
    while t < total:
        t += interval + random.uniform(-jitter, jitter)
        if dead_at is not None and t >= dead_at:
            break
        if any(s <= t < s + d for s, d in pauses):
            continue
        beats.append(t)
    return beats

traces = {
    'node-A (healthy)':       stream(),
    'node-B (1s GC pause)':   stream(pauses=[(15.0, 1.0)]),
    'node-C (1.8s partition)':stream(pauses=[(20.0, 1.8)]),
    'node-D (crash @25s)':    stream(dead_at=25.0),
    'node-E (slow but OK)':   stream(interval=1.5, jitter=0.5),
}
for name, beats in traces.items():
    print(f'{name:30s} {len(beats):3d} heartbeats, last at t={beats[-1]:.2f}s' if beats else f'{name}: no beats')

## 3. Run the detectors and define three action thresholds

From the **same** phi series we derive three layered decisions, each calibrated to how expensive a wrong call is:

| Decision | Threshold | Why |
|---|---|---|
| 🟡 **stop routing new reads** here | `φ > 3` | cheap to undo — worst case, we burn a tiny bit of latency |
| 🟠 **remove from service discovery** | `φ > 8` | Cassandra's default — reasonably sure |
| 🔴 **trigger leader fencing / failover** | `φ > 12` | Akka's default — almost certain before we shoot the other node in the head |

In [ ]:
import matplotlib.pyplot as plt

STEP = 0.05

def run(trace, total=TOTAL, step=STEP):
    det = PhiAccrual(min_std=0.3)  # tolerance for a 0.3s-jitter network
    ts, phis = [], []
    i, t = 0, 0.0
    while t < total:
        while i < len(trace) and trace[i] <= t:
            det.heartbeat(trace[i]); i += 1
        ts.append(t); phis.append(det.phi(t))
        t += step
    return ts, phis

results = {name: run(beats) for name, beats in traces.items()}

fig, ax = plt.subplots(figsize=(11, 4.5))
for name, (ts, phis) in results.items():
    ax.plot(ts, phis, label=name, linewidth=1.2)
for th, color, label in [(3, 'gold', 'route-away φ=3'), (8, 'tab:red', 'evict φ=8'), (12, 'purple', 'fence φ=12')]:
    ax.axhline(th, color=color, linestyle='--', alpha=0.6, label=label)
ax.set_ylim(0, 20)
ax.set_xlabel('time (s)'); ax.set_ylabel('φ')
ax.set_title('Phi per node across healthy / GC pause / partition / crash / slow')
ax.legend(fontsize=8, loc='upper left', ncol=2); ax.grid(True, alpha=0.2)
plt.show()

## 4. Turn the curves into decisions

For each node, for each threshold, print *when* we would have acted and whether it was the right call.

In [ ]:
def first_cross(ts, phis, th):
    for t, p in zip(ts, phis):
        if p > th:
            return t
    return None

print(f'{"node":32s} {"route-away φ>3":>16s} {"evict φ>8":>12s} {"fence φ>12":>12s}')
print('-' * 76)
for name, (ts, phis) in results.items():
    def fmt(t):
        return f'{t:6.2f}s' if t is not None else '   —   '
    print(f'{name:32s} {fmt(first_cross(ts, phis, 3)):>16s} {fmt(first_cross(ts, phis, 8)):>12s} {fmt(first_cross(ts, phis, 12)):>12s}')

### What to look for in the output

- **node-A** (healthy): no crossing at any threshold — ideal.
- **node-B** (1s GC pause): crosses `φ=3` only → we **route traffic away**, but the node comes back before we do anything destructive.
- **node-C** (1.8s partition): crosses `φ=3` and `φ=8` → we'd **evict it from membership**, but we would **not** trigger leader fencing. In a strongly-consistent cluster, eviction would later be reversed by a gossip reconcile.
- **node-D** (real crash): crosses **all three** thresholds in sequence. That monotonic escalation is exactly what you want — cheap action first, expensive action last.
- **node-E** (slow but alive): the detector *learns* its slower cadence — phi stays low. A fixed timeout tuned for 1s would have declared it dead forever.

This is the real power of phi: **one signal → many decisions**, each calibrated to the cost of being wrong.

> **Tuning note.** We used `min_std=0.3` above to model a WAN with real jitter. Cassandra defaults to `0.1` on a LAN. If you deploy a JVM service with GC pauses over a noisy network, expect to raise `min_std`, or the `phi_convict_threshold`, or both — otherwise a GC pause triggers eviction.


## 5. Wiring it into a real system (mental model)

```
                   ┌──────────────────────────────────────────┐
   heartbeats ───▶ │  PhiAccrual per peer (sliding window)    │ ──▶ φ(peer, now)
                   └──────────────────────────────────────────┘
                                       │
         ┌─────────────────────────────┼──────────────────────────────┐
         ▼                             ▼                              ▼
    router.mark_slow(peer)     membership.evict(peer)         fencing.force_new_leader()
        (φ > 3)                       (φ > 8)                        (φ > 12)
```

**Design rules of thumb**

1. **One detector per peer**, not one global detector — each link has its own cadence.
2. Call `phi(now)` on every tick (e.g. every 100 ms), not just on heartbeat arrival — otherwise a silent peer never contributes samples.
3. **Clear or re-initialize** the detector when a peer is re-added. An old window will either over- or under-react.
4. Keep the cheaper reaction (route-away) at a low threshold — it's free to undo. Reserve high thresholds for irreversible actions (fencing, eviction that triggers data rebalancing).
5. Couple phi with **other signals** for critical actions: consecutive heartbeat failures, active TCP probe, lease expiry. Phi is one vote, not the only vote.

## 6. Where phi shows up in real software

- **Apache Cassandra** — `FailureDetector.java`, default threshold `phi_convict_threshold: 8`.
- **Akka Cluster** — `akka.cluster.failure-detector.threshold = 12` by default.
- **ScyllaDB** — C++ port of Cassandra's detector.
- **Hazelcast** — offers phi as one of several failure detector options.
- **Eureka** — *does not* use phi (it uses a lease-renewal / self-preservation model) — good contrast to study.

## ✅ Recap of the whole lab

1. **Notebook 1** — a single timeout is a bad knob: short = noisy, long = slow.
2. **Notebook 2** — replace the boolean with a smooth φ driven by the network's own statistics.
3. **Notebook 3** — one φ per peer powers multiple decisions at different confidence levels, the way Cassandra/Akka actually use it.

**Further reading**

- Hayashibara, Défago, Yared & Katayama, *"The φ accrual failure detector"* (2004) — the original paper.
- Cassandra source: `src/java/org/apache/cassandra/gms/FailureDetector.java`.
- Akka docs: *Cluster Membership Service → Failure Detector*.